# A Larger Model, Speaker Context and a Memorisation Check / 更大的模型、发言背景与记忆检查

`08` 的组合（5c：DistilBERT + TF-IDF + Haiku）在 LIAR test 上达到 AUC 0.73；`09` 的错误分析发现最大的瓶颈是 **缺少发言背景**。这个 notebook 记录接下来的三个实验和一个检查：

| 步骤 | 做法 | 要回答的问题 |
|---|---|---|
| **5d** | 用更大的 Sonnet 5.5 替换或加入 Haiku | 知识更多的模型能否让组合继续提高？ |
| **记忆检查** | 只给陈述，问模型"这是谁说的" | 大模型的提升是知道更多 *事实*，还是 *记得* PolitiFact 的结论？ |
| **5f** | 告诉 Haiku 说话人、职务、州和场合（不给党派） | "缺少发言背景"是不是瓶颈？ |
| **5e** | 把 5b（维基百科证据）放进组合 | 证据能否为组合增加信息？ |

**所有步骤用同一条事先定好的规则**：组合只在 LIAR val 上训练和比较；只有 val 上的 5 折交叉验证 AUC 提高 **至少 0.01**，才在 test 上评估一次。代码：`src/ensemble.py`、`src/llm_factcheck.py`、`src/memorisation.py`。

Three experiments and one check, all under the same rule: combinations are fitted and compared on LIAR val only, and evaluated once on test only if the val CV AUC gain is at least 0.01.

## 1. Setup / 环境

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path.cwd().parent))

from src.analysis import paired_bootstrap_auc_diff
from src.baselines import evaluate
from src.error_analysis import statement_features
from src.external import load_liar, load_liar_metadata

LLM = Path('../results/llm')
pd.set_option('display.width', 220, 'display.max_colwidth', 120)

liar = load_liar().set_index('doc_id')
meta = load_liar_metadata().set_index('doc_id')

def probs(name, key='LIAR test'):
    d = np.load(LLM / f'{name}.npz', allow_pickle=True)
    return pd.Series(d[f'prob::{key}'], index=d[f'doc_id::{key}'].astype(str))

def diff(y, a, b):
    r = paired_bootstrap_auc_diff(y, a, b)
    return f"{r['AUC difference (A - B)']:+.3f} {r['95% CI']}"

haiku = probs('llm_no_evidence')
ids = haiku.index
y = liar.loc[ids, 'label']

## 2. Step 5d: a larger model / 第 5d 步：更大的模型

Sonnet 5.5 使用和 Haiku 5a **完全相同** 的提示词和输出格式。两处设置不同（否则 API 返回错误）：关闭扩展思考（`thinking: between_tools`），不传 `temperature`（Sonnet 5.5 不接受非默认值，所以它的回答有少量随机性，Haiku 是 temperature 0）。

Sonnet 5.5 gets exactly the Haiku 5a prompt and format; extended thinking is off and temperature is not sent (rejected on this model), so its answers carry slight randomness.

In [2]:
sel = json.loads((LLM / 'combination_sonnet_val.json').read_text())
print('Rule:', sel['rule'])
print('Single models, val AUC:', sel['single_model_val_auc'])
print(f"Best with Sonnet: {sel['best_with_sonnet']} | gain over 5c: {sel['gain_over_reference']} | run on test: {sel['run_test']}\n")
pd.DataFrame(sel['cv_auc']).T.sort_values('mean CV AUC', ascending=False)

Rule: run Sonnet on test only if the best Sonnet combination beats the reference by at least 0.01 mean 5-fold CV AUC on LIAR val
Single models, val AUC: {'haiku': 0.6713, 'sonnet': 0.7657}
Best with Sonnet: DistilBERT + TF-IDF + Haiku + Sonnet | gain over 5c: 0.054 | run on test: True



,seed 42,seed 43,seed 44,mean CV AUC
DistilBERT + TF-IDF + Haiku + Sonnet,0.7990,0.7945,0.7921,0.7952
DistilBERT + TF-IDF + Sonnet,0.7957,0.7917,0.7892,0.7922
Sonnet,0.7660,0.7660,0.7660,0.7660
DistilBERT + TF-IDF + Haiku,0.7457,0.7401,0.7379,0.7412
Haiku,0.6658,0.6658,0.6658,0.6658


In [3]:
sonnet = probs('llm_no_evidence_sonnet').loc[ids]
rows = {name: evaluate(y, p.to_numpy()) for name, p in (('Haiku 5a', haiku), ('Sonnet 5d', sonnet))}
print('Sonnet - Haiku (single models, test):', diff(y, sonnet, haiku))
pd.DataFrame(rows).T[['accuracy %', 'balanced accuracy %', 'AUC', 'predicted real %']]

Sonnet - Haiku (single models, test): +0.086 (0.059, 0.115)


,accuracy %,balanced accuracy %,AUC,predicted real %
Haiku 5a,61.1,62.3,0.655,42.4
Sonnet 5d,67.2,68.1,0.742,45.4


In [4]:
t5d = json.loads((LLM / 'combination_sonnet_test.json').read_text())
assert t5d['chosen_on_val'] == sel['chosen']
pd.DataFrame({seed: {'weights': r['weights (standardised logits)'],
                     'combined AUC': r['combined']['AUC'], 'combined accuracy %': r['combined']['accuracy %'],
                     'vs 5c combination': f"{r['combined - reference']['AUC difference (A - B)']:+.3f} {tuple(r['combined - reference']['95% CI'])}",
                     'vs DistilBERT': f"{r['combined - DistilBERT']['AUC difference (A - B)']:+.3f} {tuple(r['combined - DistilBERT']['95% CI'])}"}
              for seed, r in t5d['results'].items()}).T

,weights,combined AUC,combined accuracy %,vs 5c combination,vs DistilBERT
seed 42,"{'distilbert_seed42': 0.393, 'tfidf': 0.255, 'haiku': 0.224, 'sonnet': 0.945}",0.773,70.1,"+0.045 (0.029, 0.062)","+0.090 (0.067, 0.116)"
seed 43,"{'distilbert_seed43': 0.266, 'tfidf': 0.356, 'haiku': 0.221, 'sonnet': 0.942}",0.774,70.1,"+0.045 (0.028, 0.061)","+0.087 (0.061, 0.114)"
seed 44,"{'distilbert_seed44': 0.205, 'tfidf': 0.387, 'haiku': 0.233, 'sonnet': 0.949}",0.771,69.9,"+0.046 (0.03, 0.063)","+0.096 (0.071, 0.123)"


- **Sonnet 单独**在 test 上 AUC 0.742，比 Haiku（0.655）可靠地高 0.086，也超过了 `08` 里 Haiku 和文字模型的组合（0.73）。
- **5d 组合**（DistilBERT + TF-IDF + Haiku + Sonnet）AUC 0.771–0.774、准确率约 70%：比 5c 组合可靠地高 0.045，比 DistilBERT 可靠地高约 0.09（3 个种子的置信区间都不包含 0）。
- 组合器给 Sonnet 的权重（约 0.94）远大于其他模型，说明提升主要来自 Sonnet 的知识。

**Sonnet 的回答有少量随机性**（没有 temperature 0）：对 40 条 val 陈述重新提问，两次回答平均相差 5.3 个百分点，40 条中 36 条落在 50 的同一侧，排序的 Spearman 相关 0.949。AUC 只看排序，所以影响很小。

**执行中的一个问题**：test 批次运行时 API 余额耗尽，1,283 条中 86 条失败。失败的请求如果按 0.5 计入会拉低 Sonnet 的分数，所以只重新提交了这 86 条（参数完全相同），再和原批次合并（`retry_failed`）；最终没有一条缺失。

Sonnet alone reaches test AUC 0.742 (+0.086 over Haiku); the 5d combination reaches 0.771–0.774 and ~70% accuracy, reliably above 5c (+0.045) and DistilBERT (~+0.09). Sonnet's answers vary slightly between calls (Spearman 0.949 on 40 repeats). 86 test requests that failed when the API balance ran out were resubmitted unchanged and merged.

## 3. Memorisation check: has the model seen these statements? / 记忆检查

LIAR 来自 2007–2016 年公开的 PolitiFact 网页，很可能在大模型的训练数据里。Sonnet 的大幅提升可能来自知道更多 **事实**，也可能来自 **记得** 结论；后者在新的说法上不会成立。

**说话人回忆测试**（`src/memorisation.py`）：只给陈述，问"这是谁说的"。只测试说话人是个人、而且名字 **没有出现在陈述里** 的 971 条 test 陈述。回答必须包含说话人的全名才算对。能从一句原话说出一个地方小政客的名字，说明模型很可能见过这句话。

然后比较：在 **两个模型都没认出说话人** 的陈述上，Sonnet 是否仍然比 Haiku 好？如果是，它的优势主要来自知识和推理，而不是记忆。

A speaker-recall probe on 971 test statements whose speaker is a person not named in the text. If Sonnet still beats Haiku on statements neither model can attribute, its advantage is not mainly memory.

In [5]:
probe = {m: pd.DataFrame(json.loads((LLM / f'probe_speaker_{m}.json').read_text())['rows']).T
         for m in ('haiku', 'sonnet')}
pids = probe['haiku'].index
recalled = {m: probe[m].loc[pids, 'correct'].astype(bool) for m in probe}
top = meta.loc[pids, 'speaker'].value_counts()
print(f'{len(pids)} statements | always answering the most common speaker ({top.index[0]}) would score {100 * top.iloc[0] / len(pids):.1f}%')
print({m: f'{100 * r.mean():.1f}% recalled' for m, r in recalled.items()})
print('Most common answers (Sonnet):', probe['sonnet']['answer'].value_counts().head(6).to_dict())

971 statements | always answering the most common speaker (barack-obama) would score 5.8%
{'haiku': '11.8% recalled', 'sonnet': '23.1% recalled'}
Most common answers (Sonnet): {'Barack Obama': 139, 'Donald Trump': 87, 'Mitt Romney': 54, 'Hillary Clinton': 38, 'Rick Scott': 30, 'Scott Walker': 30}


In [6]:
yp = y.loc[pids]
groups = {'all probe statements': pd.Series(True, index=pids),
          'neither model recalled the speaker': ~recalled['haiku'] & ~recalled['sonnet'],
          'Sonnet recalled the speaker': recalled['sonnet']}
rows = {}
for g, m in groups.items():
    m = m.to_numpy()
    rows[g] = {'n': int(m.sum()), 'real %': round(100 * yp[m].mean(), 1),
               'Haiku AUC': round(roc_auc_score(yp[m], haiku.loc[pids][m]), 3),
               'Sonnet AUC': round(roc_auc_score(yp[m], sonnet.loc[pids][m]), 3),
               'Sonnet - Haiku': diff(yp[m], sonnet.loc[pids][m], haiku.loc[pids][m])}
pd.DataFrame(rows).T

,n,real %,Haiku AUC,Sonnet AUC,Sonnet - Haiku
all probe statements,971,58.8,0.633,0.723,"+0.090 (0.056, 0.125)"
neither model recalled the speaker,727,56.0,0.617,0.703,"+0.086 (0.047, 0.127)"
Sonnet recalled the speaker,224,67.4,0.661,0.797,"+0.136 (0.068, 0.205)"


- Sonnet 能认出的说话人（23.1%）是 Haiku（11.8%）的两倍，远高于"总猜奥巴马"的 5.8%：它"见过"的陈述确实更多。
- **但在两个模型都认不出说话人的 727 条陈述上（占 75%），Sonnet 仍然比 Haiku 可靠地高约 0.086**，和全部陈述上的差距几乎一样。所以 Sonnet 的优势 **主要来自更多的知识和更强的推理，而不是记得这些陈述**。
- 在 Sonnet 认出说话人的陈述上，差距更大（约 +0.14）。这些多是奥巴马、特朗普等名人的陈述，本来也更容易判断，所以不能确定多出来的部分是"记忆"还是"知识"。保守地说：**不能排除一小部分提升来自对这些陈述的熟悉**。
- 这个检查衡量的是"是否见过这句话"，不能直接证明模型记得 PolitiFact 的 *评级*。

Sonnet attributes twice as many statements as Haiku (23% vs 12%), but on the 75% that neither model attributes it still leads by ~0.086, about its overall lead: its advantage is mainly knowledge, not memory. A small familiarity component on recognised (mostly famous) speakers cannot be ruled out.

## 4. Step 5f: telling the model who said it / 第 5f 步：告诉模型是谁说的

`09` 发现：含"他""这项法案""我的对手"等指代的陈述上，Haiku 的 AUC 只有 0.568。5f 在陈述前加上发言背景，例如：

```
Speaker: Vicky Hartzler (U.S. Representative, Missouri)
Setting: an interview with ABC17 News
Statement: We have less Americans working now than in the 70s.
```

提示词只多了半句 "together with who made it and where"，其余和 5a 完全相同。**不给党派**（党派标签本身就是"按来源判断"的捷径），也不给历史核查次数（包含当前标签）。

**对照组**：给每条陈述 *另一条* 陈述的说话人和场合（和 5b 的对照组同样的错位分配）。

**捷径基线**：说话人在 LIAR *训练集* 中说真话的比例（向总体比例收缩，m=10，事先固定），完全不看陈述内容。它衡量"只靠认人"能得多少分。

5f adds the speaker, job, state and setting (no party) before the statement. The control uses another statement's context; the shortcut baseline is the speaker's true rate in LIAR train, ignoring the statement.

In [7]:
context = probs('llm_context').loc[ids]
shuffled = probs('llm_shuffled_context').loc[ids]
prior = probs('speaker_prior').loc[ids]
P5f = {'Haiku 5a: no context': haiku, 'control: another statement\'s context': shuffled,
       'Haiku 5f: own context': context, 'speaker prior (shortcut, no text)': prior}
pd.DataFrame({k: {m: evaluate(y, p.to_numpy())[m] for m in ('accuracy %', 'AUC', 'predicted real %')}
              for k, p in P5f.items()}).T

,accuracy %,AUC,predicted real %
Haiku 5a: no context,61.1,0.655,42.4
control: another statement's context,60.8,0.631,49.4
Haiku 5f: own context,65.3,0.680,57.8
"speaker prior (shortcut, no text)",62.2,0.649,80.6


In [8]:
pd.DataFrame({'5f - 5a': [diff(y, context, haiku)],
              'control - 5a': [diff(y, shuffled, haiku)],
              '5f - control': [diff(y, context, shuffled)]}, index=['AUC difference (95% CI), test']).T

,"AUC difference (95% CI), test"
5f - 5a,"+0.024 (0.004, 0.045)"
control - 5a,"-0.024 (-0.045, -0.002)"
5f - control,"+0.049 (0.026, 0.069)"


和 5b 一样，对照组把变化拆成两部分：换了提示格式（而且背景对不上）让 AUC 下降约 0.024；背景内容对得上，带来约 +0.049。**和 5b 不同的是，净效果是正的**：5f 比 5a 可靠地高 0.024。

但 val 上 5f 只比 5a 高 0.003。把 val 和 test 合在一起（2,567 条）可以更准确地估计效果的大小（这只是估计，不选择任何东西）：

In [9]:
y_all = liar['label']
a_all = pd.concat([probs('llm_no_evidence_val', 'LIAR val'), haiku])
f_all = pd.concat([probs('llm_context_val', 'LIAR val'), context]).loc[a_all.index]
n_val = len(a_all) - len(haiku)
pd.DataFrame({'val': [diff(y_all[a_all.index[:n_val]], f_all[:n_val], a_all[:n_val])],
              'test': [diff(y_all[a_all.index[n_val:]], f_all[n_val:], a_all[n_val:])],
              f'val + test (n={len(a_all)})': [diff(y_all[a_all.index], f_all, a_all)]},
             index=['5f - 5a AUC difference (95% CI)']).T

,5f - 5a AUC difference (95% CI)
val,"+0.003 (-0.02, 0.025)"
test,"+0.024 (0.004, 0.045)"
val + test (n=2567),"+0.014 (-0.002, 0.029)"


合并后约 **+0.014，置信区间包含 0**：val 和 test 的结果并不矛盾，真实的效果大约是 +0.01，两个各约 1,300 条的数据集因为随机波动一个偏低、一个偏高。**告诉模型说话人，效果很小，不能可靠地确认。**

Pooled over val and test the effect is about +0.014 with a CI including 0: small and not reliably positive.

但"有没有提高"还不够，还要看 **为什么** 提高。

### 4.1 Where does the gain come from? / 提升来自哪里

如果说话人信息真的在 **补背景**（让模型知道"他""这项法案"指什么），提升应该 **集中在含指代的陈述上**。如果模型只是在 **认人**（"这个人经常说谎"或者记得这个人的核查结果），提升应该集中在 **知名的说话人** 上，而且模型分数的变化会跟着说话人的名声走。

In [10]:
F = statement_features(liar.loc[ids, 'text'])
L = load_liar(); M = load_liar_metadata()
train_count = M.loc[L['split'].values == 'train', 'speaker'].value_counts()
n_train = meta.loc[ids, 'speaker'].map(train_count).fillna(0)
groups = {'no context-dependent reference': ~F['context-dependent reference'],
          'context-dependent reference ("he", "this bill")': F['context-dependent reference'],
          'speaker rare in train (< 5 statements)': n_train < 5,
          'speaker in train 5-49 times': (n_train >= 5) & (n_train < 50),
          'prominent speaker (>= 50 in train)': n_train >= 50}
rows = {}
for g, m in groups.items():
    m = m.to_numpy()
    rows[g] = {'n': int(m.sum()), 'AUC 5a': round(roc_auc_score(y[m], haiku[m]), 3),
               'AUC 5f': round(roc_auc_score(y[m], context[m]), 3),
               'AUC speaker prior': round(roc_auc_score(y[m], prior[m]), 3),
               '5f - 5a': diff(y[m], context[m], haiku[m])}
pd.DataFrame(rows).T

,n,AUC 5a,AUC 5f,AUC speaker prior,5f - 5a
no context-dependent reference,1009,0.677,0.702,0.651,"+0.025 (0.001, 0.049)"
"context-dependent reference (""he"", ""this bill"")",274,0.568,0.591,0.642,"+0.023 (-0.033, 0.076)"
speaker rare in train (< 5 statements),456,0.665,0.694,0.555,"+0.029 (-0.006, 0.064)"
speaker in train 5-49 times,466,0.657,0.653,0.678,"-0.005 (-0.037, 0.027)"
prominent speaker (>= 50 in train),361,0.634,0.692,0.679,"+0.058 (0.018, 0.101)"


In [11]:
logit = lambda p: np.log(np.clip(p, .01, .99) / (1 - np.clip(p, .01, .99)))
print('Does the change in Haiku\'s score follow the speaker\'s reputation? (Spearman with the speaker prior)')
print(f"  5f - 5a:      {spearmanr(logit(context) - logit(haiku), prior)[0]:.3f}")
print(f"  control - 5a: {spearmanr(logit(shuffled) - logit(haiku), prior)[0]:.3f}   (reference: should be ~0)")

Does the change in Haiku's score follow the speaker's reputation? (Spearman with the speaker prior)
  5f - 5a:      0.120
  control - 5a: -0.012   (reference: should be ~0)


- **提升没有集中在含指代的陈述上**：两类陈述提升差不多（约 +0.02），含指代的陈述上 5f 的 AUC 仍然只有约 0.59。告诉模型说话人的名字，并没有补上"这项法案是什么、什么时候说的"这类背景。
- **提升集中在知名说话人上**：出现 50 次以上的说话人提升约 +0.06（可靠），中等知名度的说话人没有提升。
- **分数的变化跟着说话人的名声走**（相关系数约 0.12，对照组约 0）。

所以 5f 的小幅提升 **更像"认出了知名人物"（名声或记忆），而不是"理解了背景"**。

The gain is not concentrated on context-dependent statements (both ~+0.02; AUC there stays ~0.59) but on prominent speakers (+0.06), and Haiku's score shift follows the speaker's reputation: 5f looks like recognising famous speakers, not understanding context.

In [12]:
s5f = json.loads((LLM / 'combination_context_val.json').read_text())
print('Rule:', s5f['rule'])
for q, info in s5f['questions'].items():
    print(f"  {q}: best = {info['best']} | gain = {info['gain']:+.4f} | evaluate on test: {info['run_test']}")
pd.DataFrame(s5f['cv_auc']).T.sort_values('mean CV AUC', ascending=False)

Rule: evaluate on test only if a question gains at least 0.01 mean 5-fold CV AUC on LIAR val
  Q1: improves the 5c combination: best = DistilBERT + TF-IDF + Haiku + Haiku context | gain = +0.0034 | evaluate on test: False
  Q2: adds to the 5d Sonnet combination: best = DistilBERT + TF-IDF + Haiku + Sonnet + Haiku context | gain = -0.0015 | evaluate on test: False


,seed 42,seed 43,seed 44,mean CV AUC
DistilBERT + TF-IDF + Haiku + Sonnet + speaker prior (shortcut),0.8043,0.8006,0.7988,0.8012
DistilBERT + TF-IDF + Haiku + Sonnet,0.7990,0.7945,0.7921,0.7952
DistilBERT + TF-IDF + Haiku + Sonnet + Haiku context,0.7974,0.7930,0.7906,0.7937
DistilBERT + TF-IDF + Haiku + speaker prior (shortcut),0.7568,0.7520,0.7500,0.7529
DistilBERT + TF-IDF + Haiku + Haiku context,0.7491,0.7435,0.7413,0.7446
DistilBERT + TF-IDF + Haiku,0.7457,0.7401,0.7379,0.7412
DistilBERT + TF-IDF + Haiku context,0.7407,0.7354,0.7324,0.7362
Haiku + context,0.6704,0.6704,0.6704,0.6704
Haiku,0.6658,0.6658,0.6658,0.6658
speaker prior (shortcut),0.6334,0.6334,0.6334,0.6334


**在组合中，5f 没有达到规则**：在 5c 的组合上只提高 0.003，在 Sonnet 的组合上反而 −0.002，所以 **没有在 test 上评估**。

一个说明问题的对比：把纯粹的 **说话人名声基线**（不看陈述）加进组合，在 val 上能提高约 0.012，**比把 5f 加进去还多**。也就是说，5f 中 Haiku 从说话人信息里得到的，大部分只是名声这一类信息，而名声本身就是我们一直要避免的"按来源判断"的捷径。名声基线 **只用于诊断，不会被选进模型**。

In the combination 5f fails the rule (+0.003 / −0.002) and is not evaluated on test. Adding the pure reputation baseline would add more (+0.012) than 5f does, so what Haiku extracts from the speaker is mostly reputation, the shortcut this project avoids. The baseline is diagnostic only.

## 5. Step 5e: Wikipedia evidence inside the combination / 第 5e 步：把维基百科证据放进组合

5b 单独使用时没有超过 5a（`08`），但 5b 和 5a 的打分相关系数只有 0.62：看了证据之后，Haiku 改变了对不少陈述的判断。5e 检验这些"不同的看法"能否为组合增加信息。

做法：为 LIAR val 用 **完全相同的流程** 检索维基百科证据（LLM 写检索词 → 维基百科 → 防泄漏过滤 → 选 3 段），再在 val 上跑 5b。test 上的 5b 结果在 `08` 中已经有了。

**val 证据的审计**：不看标签扫描了全部 3,847 段。发现 1 条陈述（内容本身就是关于 PolitiFact"年度谎言"的）检索到了标题为 "PolitiFact" 的文章，段落列出的是 2022、2024 年的"年度谎言"，没有透露这条陈述的结论。为了规则一致，防泄漏过滤现在也检查 **文章标题**，并为这一条重新检索；test 的证据中没有任何标题受这条规则影响，保持不变。另有 4 段以无关的方式出现了 "false claims"（例如关于 Facebook 的诉讼），不是对陈述的判断。

Evidence for LIAR val was retrieved with the identical pipeline. A label-blind scan found one statement (itself about PolitiFact's Lie of the Year) whose passages came from the article "PolitiFact"; they did not reveal its verdict, but the leak filter now also checks article titles and that statement was re-retrieved. No test passage is affected by the new rule.

In [13]:
evidence = probs('llm_evidence').loc[ids]
evidence_val = probs('llm_evidence_val', 'LIAR val')
haiku_val = probs('llm_no_evidence_val', 'LIAR val').loc[evidence_val.index]
y_val = liar.loc[evidence_val.index, 'label']
print(f"5b vs 5a, val:  AUC {roc_auc_score(y_val, evidence_val):.3f} vs {roc_auc_score(y_val, haiku_val):.3f} | difference {diff(y_val, evidence_val, haiku_val)}")
print(f"5b vs 5a, test: AUC {roc_auc_score(y, evidence):.3f} vs {roc_auc_score(y, haiku):.3f} | difference {diff(y, evidence, haiku)}")
print(f"Spearman correlation of 5b with 5a: val {spearmanr(evidence_val, haiku_val)[0]:.2f}, test {spearmanr(evidence, haiku)[0]:.2f}")
s5e = json.loads((LLM / 'combination_evidence_val.json').read_text())
for q, info in s5e['questions'].items():
    print(f"  {q}: best = {info['best']} | gain = {info['gain']:+.4f} | evaluate on test: {info['run_test']}")
pd.DataFrame(s5e['cv_auc']).T.sort_values('mean CV AUC', ascending=False)

5b vs 5a, val:  AUC 0.659 vs 0.671 | difference -0.012 (-0.038, 0.012)


5b vs 5a, test: AUC 0.647 vs 0.655 | difference -0.008 (-0.034, 0.018)
Spearman correlation of 5b with 5a: val 0.67, test 0.62
  Q1: improves the 5c combination: best = DistilBERT + TF-IDF + Haiku + Haiku evidence | gain = +0.0035 | evaluate on test: False
  Q2: adds to the 5d Sonnet combination: best = DistilBERT + TF-IDF + Haiku + Sonnet + Haiku evidence | gain = -0.0004 | evaluate on test: False


,seed 42,seed 43,seed 44,mean CV AUC
DistilBERT + TF-IDF + Haiku + Sonnet,0.7990,0.7945,0.7921,0.7952
DistilBERT + TF-IDF + Haiku + Sonnet + Haiku evidence,0.7985,0.7941,0.7917,0.7948
DistilBERT + TF-IDF + Haiku + Haiku evidence,0.7490,0.7440,0.7412,0.7447
DistilBERT + TF-IDF + Haiku,0.7457,0.7401,0.7379,0.7412
Haiku,0.6658,0.6658,0.6658,0.6658
Haiku + evidence,0.6522,0.6522,0.6522,0.6522


- val 上 5b 同样低于 5a（0.659 对 0.671），和 test 上的模式一致。
- **放进组合后也没有帮助**：在 5c 的组合上只提高 0.0035，在 Sonnet 的组合上 −0.0004，都没有达到 0.01，所以 **没有在 test 上评估**。
- 5b 和 5a 的 *排序* 确实不同（Spearman 约 0.6，排序相关不受整体偏移影响），但证据带来的这些排序变化 **并没有更接近真相**：5b 的 AUC 反而更低。和 Haiku 相比 Sonnet 的不同是"知道更多"，而 5b 的不同主要是"被不相关的段落带偏"，所以组合器从中得不到多少新信息。

5b is below 5a on val too (0.659 vs 0.671); in the combination it adds +0.0035 (5c) and −0.0004 (5d), below the 0.01 rule, so it is not evaluated on test. Evidence does reorder the statements (rank correlation ~0.6), but the new order is not closer to the truth, so it offers the combiner little new information.

## 6. Conclusions / 结论

| LIAR test（1,283 条，全部猜真 = 56.7%） | 准确率 | AUC | 在 test 上评估？ |
|---|---:|---:|---|
| DistilBERT（3 个种子，`07`） | 63.5–65.0% | 0.675–0.687 | 是 |
| 5c：DistilBERT + TF-IDF + Haiku（`08`） | 66.4–67.5% | 0.725–0.729 | 是 |
| Sonnet 5.5 单独（5d） | 67.2% | 0.742 | 是 |
| **5d：DistilBERT + TF-IDF + Haiku + Sonnet** | **69.9–70.1%** | **0.771–0.774** | 是（val +0.054） |
| 5f：Haiku + 说话人和场合 | 65.3% | 0.680 | 单个模型是；组合没有达到规则（val +0.003） |
| 5e：组合中加入 5b（维基百科证据） | — | — | 没有达到规则（val +0.0035） |
| 参考：只看"谁说的"（捷径，不看内容） | 62.2% | 0.649 | 只用于诊断 |

**1. 更大的模型带来了真正的提升，而且主要不是靠记忆。** Sonnet 单独就比 Haiku 高 0.086；5d 组合达到 AUC 约 0.77、准确率约 70%，比 5c 可靠地高 0.045，比 DistilBERT 高约 0.09。Sonnet 能认出的说话人是 Haiku 的两倍（23% 对 12%），但在两个模型都认不出说话人的 75% 陈述上，它仍然高 0.086。

**2. 只告诉模型"谁说的"，补不上缺失的背景，反而打开了"认人"的捷径。** 5f 的效果很小（val 和 test 合并 +0.014，置信区间包含 0），提升集中在知名政客上，而不是含"他""这项法案"的陈述上，分数的变化也跟着说话人的名声走。只看说话人的名声，不看内容，AUC 就有 0.649：这和 ISOT 上"认来源"是同一类捷径，所以 5d 组合中 **没有** 加入任何说话人信息。

**3. 维基百科证据既不能单独提升，也不能为组合增加信息**（5e：+0.0035 / −0.0004）。

**所以，在 LIAR 上真正有效的是"语言模式 + 更多的世界知识"**：文字模型和一个知识更多的大模型组合起来。外部证据和说话人信息都没有带来可靠的帮助。

**局限**
- 记忆检查衡量的是"是否见过这句话"，不能直接证明模型记得评级；在 Sonnet 认出说话人的陈述上，它的优势更大（+0.14），不能排除一小部分来自熟悉。
- Sonnet 没有 temperature 0，回答有少量随机性（重复提问的排序相关 0.949）。
- 组合器的权重和阈值都在 val 上学到；val 和 test 的真陈述比例不同（52.0% 对 56.7%）。
- 5d、5e、5f 都在同一个 test 集上报告，但每一个都是事先在 val 上决定的，没有在 test 上挑选。
- API 余额耗尽时 86 条 Sonnet test 请求和 479 条记忆检查请求失败，已用完全相同的参数重新提交并合并，没有缺失。

---

1. **A larger model helps, mainly through knowledge rather than memory**: Sonnet alone beats Haiku by 0.086 AUC, and DistilBERT + TF-IDF + Haiku + Sonnet reaches test AUC 0.771–0.774 (~70% accuracy), reliably above 5c (+0.045) and DistilBERT (~+0.09). On the 75% of statements neither model can attribute to their speaker, Sonnet still leads by 0.086.
2. **Telling the model who said it does not supply the missing context**: the 5f gain is small (pooled +0.014, CI includes 0), concentrated on prominent speakers and aligned with reputation. Reputation alone reaches AUC 0.649, a shortcut of the same kind as ISOT's source, so no speaker information enters the final model.
3. **Wikipedia evidence adds nothing even inside the combination** (5e: +0.0035 / −0.0004 on val).